# FarmTech Solutions — Assistente Agrícola Inteligente (Fase 4, Cap 1)

**Autor:** Jefferson Gonçalves Lemos · **RM:** 572399 · **Fase:** 4 · **Capítulo:** 1

**Disciplina:** Inteligência Artificial — FIAP

---

Este notebook segue a metodologia **CRISP-DM** aplicada a um problema de **regressão** (não classificação). O objetivo é construir modelos de Machine Learning supervisionado capazes de **prever variáveis agronômicas contínuas** a partir de leituras de sensores de solo e clima coletadas por pivôs de irrigação (`p1`, `p2`, `p3`).

As fases do CRISP-DM cobertas são: Entendimento do Negócio, Entendimento dos Dados, Preparação dos Dados, Modelagem, Avaliação e (preparação para) Implantação.

## 1. Configuração e Imports

Importação das bibliotecas de manipulação de dados (`pandas`, `numpy`), visualização (`matplotlib`, `seaborn`), Machine Learning (`scikit-learn`) e persistência de modelos (`joblib`). Definimos também as constantes de caminho e a *seed* global para garantir **reprodutibilidade** (`random_state=42`).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo de visualização
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Caminhos do projeto (relativos à raiz fase4cap1)
BASE_DIR = Path('..').resolve()
RAW_CSV = BASE_DIR / 'data' / 'raw' / 'seed_data.csv'
PROCESSED_PARQUET = BASE_DIR / 'data' / 'processed' / 'dataset_ml.parquet'
MODELS_DIR = BASE_DIR / 'models'
METRICS_JSON = MODELS_DIR / 'metrics.json'

print('pandas', pd.__version__)
print('numpy', np.__version__)
print('Dataset bruto:', RAW_CSV)

## 2. CRISP-DM — Entendimento do Negócio

A **FarmTech Solutions** opera um sistema de irrigação inteligente em uma fazenda dividida em três pivôs (`p1`, `p2`, `p3`). Sensores capturam, ao longo do tempo, leituras de **umidade do solo, temperatura, pH e nutrientes (N, P, K)**, além do **estado da bomba** de irrigação.

**Problema de negócio:** apoiar a tomada de decisão do produtor antecipando indicadores agronômicos que hoje só são conhecidos *a posteriori* (após a colheita) ou exigem análises laboratoriais caras. Isso permite **otimizar irrigação, adubação e estimar produtividade**.

### Objetivo de ML (regressão)
Treinar um modelo de regressão para cada um dos **5 alvos contínuos**:

| Alvo | Unidade / Faixa | Origem |
|------|-----------------|--------|
| `rendimento` | sacas/ha (0–80) | Engenheirado (heurística agronômica) |
| `volume_irrigacao` | mm (0–40) | Engenheirado |
| `necessidade_fertilizacao` | índice (0–100) | Engenheirado |
| `umidade` | % | **Real** (já presente nos dados) |
| `ph` | — | **Real** (já presente nos dados) |

**Critério de sucesso:** modelos com **R² elevado** e **erro (MAE/RMSE) baixo** no conjunto de teste, superando a baseline linear, e com interpretabilidade que confirme as relações agronômicas esperadas.

## 3. CRISP-DM — Entendimento dos Dados

Carregamos o dataset bruto `data/raw/seed_data.csv` (2696 registros + cabeçalho). Inspecionamos estrutura (`info`), primeiras linhas (`head`) e estatísticas descritivas (`describe`). Verificamos tipos, valores ausentes e a cardinalidade das colunas categóricas (`ID_PIVO`, `ESTADO_BOMBA`).

In [ ]:
# Carregamento do dataset bruto
df_raw = pd.read_csv(RAW_CSV, parse_dates=['CAPTURADO_EM'])
print('Formato (linhas, colunas):', df_raw.shape)
df_raw.head()

In [ ]:
# Estrutura e tipos de dados
df_raw.info()

In [ ]:
# Estatísticas descritivas das variáveis numéricas
df_raw.describe()

In [ ]:
# Valores ausentes e cardinalidade das categóricas
print('Valores ausentes por coluna:')
print(df_raw.isna().sum())
print()
print('ID_PIVO:', df_raw['ID_PIVO'].unique())
print('ESTADO_BOMBA:', df_raw['ESTADO_BOMBA'].unique())
print('Período coberto:', df_raw['CAPTURADO_EM'].min(), '→', df_raw['CAPTURADO_EM'].max())

## 4. CRISP-DM — Preparação dos Dados

Esta etapa é executada de forma canônica pelo script **`ml/prepare_dataset.py`**, que gera `data/processed/dataset_ml.parquet`. Aqui reproduzimos a lógica de forma didática.

### 4.1 Feature engineering (entradas do modelo)
- `estado_bomba` → binário (1 = ON, 0 = OFF)
- `id_pivo` → *one-hot encoding* (`p1`, `p2`, `p3`)
- `hora` e `dia_semana` → derivadas de `capturado_em`
- features numéricas mantidas: `temperatura`, `n`, `p`, `k`

### 4.2 Engenharia dos alvos (⚠️ alvos SIMULADOS)
Os alvos `rendimento`, `volume_irrigacao` e `necessidade_fertilizacao` **não são medidos diretamente** pelos sensores. Eles são **simulados por heurísticas agronômicas** (`numpy seed 42`) usando a **função sino** `bell(x, opt, sd) = exp(-0.5*((x-opt)/sd)**2)`, que vale 1 no ótimo e decai conforme o afastamento. Os alvos `umidade` e `ph` são **reais**.

**Constantes agronômicas:** umidade ótima 62.5 (faixa 55–70, sd 12); pH ótimo 6.5 (faixa 6.0–7.0, sd 0.6); temperatura ótima 25 (faixa 20–30, sd 7); N ideal 70, P ideal 60, K ideal 60; umidade alvo de irrigação 65.

> **Transparência metodológica:** por serem simulados, os alvos engenheirados servem para demonstrar o *pipeline* completo de regressão. Em produção seriam substituídos por medições reais (produtividade colhida, vazão da bomba, laudos de solo).

In [ ]:
# Função sino (gaussiana normalizada): 1.0 no ótimo, decai com o afastamento
def bell(x, opt, sd):
    return np.exp(-0.5 * ((x - opt) / sd) ** 2)

def relu(x):
    return np.maximum(0, x)

# --- Feature engineering ---
df = df_raw.copy()
df.columns = [c.lower() for c in df.columns]
df['estado_bomba'] = (df['estado_bomba'] == 'ON').astype(int)
df['hora'] = df['capturado_em'].dt.hour
df['dia_semana'] = df['capturado_em'].dt.dayofweek
df = pd.get_dummies(df, columns=['id_pivo'], prefix='', prefix_sep='')
for p in ['p1', 'p2', 'p3']:
    if p not in df.columns:
        df[p] = 0
    df[p] = df[p].astype(int)

df.head()

In [ ]:
# --- Engenharia dos alvos simulados (numpy seed 42) ---
np.random.seed(RANDOM_STATE)
n_rows = len(df)

# rendimento (sacas/ha, 0-80)
f_npk = (bell(df['n'], 70, 25) + bell(df['p'], 60, 25) + bell(df['k'], 60, 25)) / 3
rendimento = (80 * bell(df['umidade'], 62.5, 12) * bell(df['ph'], 6.5, 0.6)
              * f_npk * bell(df['temperatura'], 25, 7)
              + np.random.normal(0, 3, n_rows))
df['rendimento'] = np.clip(rendimento, 0, 80)

# volume_irrigacao (mm, 0-40)
volume = (relu(65 - df['umidade']) * 0.8 + 0.15 * relu(df['temperatura'] - 20)
          + np.random.normal(0, 1.5, n_rows))
df['volume_irrigacao'] = np.clip(volume, 0, 40)

# necessidade_fertilizacao (indice 0-100)
nf = ((relu(70 - df['n']) + relu(60 - df['p']) + relu(60 - df['k'])) / (70 + 60 + 60) * 100
      + np.random.normal(0, 3, n_rows))
df['necessidade_fertilizacao'] = np.clip(nf, 0, 100)

df[['rendimento', 'volume_irrigacao', 'necessidade_fertilizacao', 'umidade', 'ph']].describe()

In [ ]:
# Definição de features e alvos — FEATURES POR-ALVO (alinhado a ml/train.py)
BASE_FEATURES = ['temperatura', 'n', 'p', 'k', 'estado_bomba', 'p1', 'p2', 'p3', 'hora', 'dia_semana']
CONTEXT_FEATURES = ['umidade', 'ph']  # leituras de sensor: features dos alvos ENGENHEIRADOS
ENGINEERED = ['rendimento', 'volume_irrigacao', 'necessidade_fertilizacao']
TARGETS = ['rendimento', 'volume_irrigacao', 'necessidade_fertilizacao', 'umidade', 'ph']

# Regra (sem data leakage): os alvos engenheirados dependem de umidade+ph por
# construção, então usam BASE + CONTEXT; os alvos reais (umidade, ph) usam só a
# BASE e nunca entram como feature de si mesmos.
print('BASE_FEATURES:', BASE_FEATURES)
print('CONTEXT_FEATURES (só p/ engenheirados):', CONTEXT_FEATURES)
print('Alvos:', TARGETS)

## 5. Análise Exploratória dos Dados (EDA)

Visualizamos a **distribuição** das variáveis (histogramas), a presença de *outliers* (boxplots) e as **correlações** entre features e alvos (matriz de correlação). A EDA orienta decisões de modelagem e ajuda a antecipar a importância de cada variável.

In [ ]:
# Histogramas das variáveis numéricas principais
cols_hist = ['umidade', 'temperatura', 'ph', 'n', 'p', 'k', 'rendimento', 'volume_irrigacao', 'necessidade_fertilizacao']
df[cols_hist].hist(bins=30, figsize=(14, 10))
plt.suptitle('Distribuição das variáveis (features e alvos)')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots para inspeção de outliers nas features de sensor
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, ['umidade', 'temperatura', 'ph', 'n']):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlação (Pearson) entre features e alvos
corr_cols = ['temperatura', 'n', 'p', 'k', 'estado_bomba', 'hora', 'dia_semana'] + TARGETS
plt.figure(figsize=(12, 9))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlação')
plt.tight_layout()
plt.show()

## 6. CRISP-DM — Modelagem

Para **cada alvo** treinamos e comparamos três algoritmos de regressão do `scikit-learn`:

1. **`LinearRegression`** — *baseline* linear simples.
2. **`Ridge`** — regressão linear com regularização L2 (controla *overfitting*).
3. **`RandomForestRegressor`** — modelo não-linear baseado em árvores, captura interações.

Usamos *split* **treino/teste 80/20** com `random_state=42`. Importante: ao prever `umidade` ou `ph`, esse alvo é removido das features para evitar **vazamento de dados** (*data leakage*).

In [ ]:
def montar_X_y(df, target):
    """Monta X e y aplicando FEATURES POR-ALVO (sem data leakage).

    Alvos engenheirados (rendimento, volume_irrigacao, necessidade_fertilizacao)
    recebem BASE + CONTEXT (umidade, ph); alvos reais (umidade, ph) usam só a BASE.
    O proprio alvo nunca entra como feature.
    """
    feats = list(BASE_FEATURES)
    if target in ENGINEERED:
        feats += CONTEXT_FEATURES  # umidade/ph sao os principais drivers desses alvos
    feats = [f for f in feats if f != target]
    X = df[feats].astype(float)
    y = df[target].astype(float)
    return X, y, feats

# Exemplo: split para o alvo 'rendimento' (agora COM umidade/ph -> R2 ~0.88)
X_r, y_r, feats_r = montar_X_y(df, 'rendimento')
X_train, X_test, y_train, y_test = train_test_split(
    X_r, y_r, test_size=0.20, random_state=RANDOM_STATE)
print('Features de rendimento (%d):' % len(feats_r), feats_r)
print('Treino:', X_train.shape, '| Teste:', X_test.shape)

In [ ]:
# Dicionário de candidatos por alvo (instância padrão de cada algoritmo)
def candidatos():
    return {
        'LinearRegression': LinearRegression(),
        'Ridge': Ridge(random_state=RANDOM_STATE),
        'RandomForest': RandomForestRegressor(random_state=RANDOM_STATE, n_estimators=200),
    }

# Treino exemplificativo dos três modelos para o alvo 'rendimento'
modelos_rend = {}
for nome, modelo in candidatos().items():
    modelo.fit(X_train, y_train)
    modelos_rend[nome] = modelo
    print(f'{nome}: R2 treino = {modelo.score(X_train, y_train):.3f}')

## 7. CRISP-DM — Avaliação

Avaliamos cada modelo no conjunto de **teste** com as métricas de regressão:

- **MAE** (Erro Absoluto Médio) — erro médio na unidade do alvo.
- **MSE** (Erro Quadrático Médio) — penaliza erros grandes.
- **RMSE** (Raiz do MSE) — erro na unidade do alvo, sensível a *outliers*.
- **R²** (Coeficiente de determinação) — fração da variância explicada (1.0 = perfeito).

Construímos uma **tabela comparativa** e o gráfico de dispersão **predito × real** do melhor modelo. O *pipeline* canônico de treino e persistência está em **`ml/train.py`**, que salva os modelos em `models/*.joblib` e as métricas em `models/metrics.json`.

In [ ]:
def avaliar(modelo, X_test, y_test):
    pred = modelo.predict(X_test)
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

# Tabela comparativa dos três modelos para 'rendimento'
linhas = []
for nome, modelo in modelos_rend.items():
    m = avaliar(modelo, X_test, y_test)
    m['Modelo'] = nome
    linhas.append(m)
tabela = pd.DataFrame(linhas)[['Modelo', 'MAE', 'MSE', 'RMSE', 'R2']]
tabela.sort_values('R2', ascending=False)

In [ ]:
# Scatter predito vs real do melhor modelo (maior R2) para 'rendimento'
melhor_nome = tabela.sort_values('R2', ascending=False).iloc[0]['Modelo']
melhor = modelos_rend[melhor_nome]
pred = melhor.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, pred, alpha=0.3, edgecolor='k', linewidth=0.2)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
plt.plot(lims, lims, 'r--', label='ideal (y = x)')
plt.xlabel('Rendimento real (sacas/ha)')
plt.ylabel('Rendimento predito (sacas/ha)')
plt.title(f'Predito vs Real — {melhor_nome}')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Otimização de Hiperparâmetros (GridSearchCV)

Refinamos o modelo não-linear (`RandomForestRegressor`) com **`GridSearchCV`** e **validação cruzada de 5 folds** (`cv=5`), otimizando por R². A busca em grade testa combinações de `n_estimators`, `max_depth` e `min_samples_split`. O mesmo procedimento é aplicado a cada alvo no script `ml/train.py`, que seleciona automaticamente o melhor estimador.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print('Melhores hiperparâmetros:', grid.best_params_)
print('Melhor R2 (CV):', round(grid.best_score_, 4))
print('R2 no teste:', round(grid.best_estimator_.score(X_test, y_test), 4))

In [ ]:
# Persistência do melhor modelo (exemplo para 'rendimento')
# MODELS_DIR.mkdir(exist_ok=True)
# joblib.dump(grid.best_estimator_, MODELS_DIR / 'modelo_rendimento.joblib')
# O treino completo dos 5 alvos e a gravação de models/metrics.json são feitos por ml/train.py
print('Para treinar e salvar todos os alvos, execute: python ml/train.py')

## 9. Interpretação e Insights Agronômicos

Inspecionamos a **importância das features** (`feature_importances_`) do RandomForest para entender quais variáveis mais influenciam cada alvo, conectando o resultado estatístico ao conhecimento agronômico:

- **Rendimento:** espera-se forte dependência de `umidade`, `temperatura` e dos nutrientes (N, P, K), refletindo as funções sino centradas nos ótimos agronômicos.
- **Volume de irrigação:** dominado pelo déficit de umidade em relação ao alvo (65) e pela temperatura (evapotranspiração).
- **Necessidade de fertilização:** governada pelos déficits de N, P e K frente aos ideais (70/60/60).

Esses padrões validam que o modelo aprendeu relações **fisicamente plausíveis**.

In [ ]:
# Importância das features do melhor RandomForest (alvo 'rendimento')
rf = grid.best_estimator_
importancias = pd.Series(rf.feature_importances_, index=feats_r).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importancias.plot(kind='barh', color='seagreen')
plt.title('Importância das features — Rendimento (RandomForest)')
plt.xlabel('Importância relativa')
plt.tight_layout()
plt.show()

importancias.sort_values(ascending=False)

## 10. Conclusão e Próximos Passos

### Síntese
- Aplicamos o ciclo **CRISP-DM** completo a um problema de **regressão multi-alvo** sobre dados de sensores agrícolas dos pivôs `p1`/`p2`/`p3`.
- Comparamos **LinearRegression**, **Ridge** e **RandomForestRegressor**; o modelo não-linear tende a vencer nos alvos com interações fortes, refinado por **GridSearchCV (cv=5)**.
- Avaliamos com **MAE, MSE, RMSE e R²** e interpretamos via `feature_importances_`, confirmando relações agronômicas plausíveis.

### Limitações
- Os alvos `rendimento`, `volume_irrigacao` e `necessidade_fertilizacao` são **simulados** por heurística; em produção devem ser substituídos por medições reais.

### Próximos passos
- Integrar os modelos versionados (`models/*.joblib`) ao **dashboard Streamlit** (`streamlit/app.py`) com fonte de dados comutável (`DATA_SOURCE` local/cloud).
- Consumir dados reais via **API FastAPI + Oracle XE** (Cap-3) na arquitetura híbrida.
- Coletar alvos reais (produtividade colhida, vazão, laudos de solo) para re-treino supervisionado.
- Publicar o dashboard no **Streamlit Cloud** (ver `docs/DEPLOY.md`).